In [47]:
import yaml
from src import paths
import pandas as pd
import numpy as np
import os


In [48]:
# from dotenv import load_dotenv


# from sqlalchemy import create_engine

# # Load environment variables from .env file
# load_dotenv("../private_data/.env")

# host = os.getenv("HOST")
# db = os.getenv("DB")
# port = os.getenv("PORT")
# role = os.getenv("ROLE")
# pw = os.getenv("PASSWORD")
# engine = create_engine(f"postgresql+psycopg2://{role}:{pw}@{host}:{port}/{db}")

# Barter deals

## Load data

In [49]:
df = pd.read_parquet(paths.RAW_DATA_DIR / 'BARTER_DEALS.parquet')
df = df.rename(columns = {'title': 'deal_title', 'description': 'deal_text'})


# Strip timezone from both columns
df['created_at'] = pd.to_datetime(df['created_at']).dt.tz_localize(None)
df['deleted_at'] = pd.to_datetime(df['deleted_at']).dt.tz_localize(None)
df['updated_at'] = pd.to_datetime(df['updated_at']).dt.tz_localize(None)

df.loc[:,'diff_created_deleted'] = df.deleted_at - df.created_at
df.loc[:,'diff_updated_deleted'] = df.updated_at - df.created_at



## Filter dirty rows

- Delete deals that were online for less than $n$ days

In [50]:
print(len(df))
df = df[~(df.diff_created_deleted < pd.Timedelta(days=7))]
print(len(df))

7277
7092


- Remove deals that are too short

In [51]:
print(len(df))
df['text_word_count'] = df['deal_text'].str.split().str.len().fillna(0)
df['title_word_count'] = df['deal_title'].str.split().str.len().fillna(0)

mask_short_text = ((df.title_word_count <= 2) & (df.text_word_count <= 5) & (df.applicants_applications_count < 10))
test = df.loc[mask_short_text,:]
df = df.loc[~mask_short_text,:]
print(len(df))

7092
7049


- remove 'duplicate' deals

In [52]:
print(len(df))
df = df[~(df['deal_title'].str.contains('duplicate') & (df.applicants_applications_count < 5))]
print(len(df))

7049
6880


- Remove deals that never went live

In [53]:
print(len(df))
df = df[~df['go_live_at'].isna()]
print(len(df))

6880
6428


- days since start column create

In [54]:
# 1. Ensure datetime format
df['created_at'] = pd.to_datetime(df['created_at'])

# 2. Extract a numeric 'Time' feature for correlation (e.g., Days since start)
# We subtract the minimum date to get a "Day 0", "Day 1"... counter
start_date = df['created_at'].min()
df['days_since_start'] = (
    df['created_at'] - start_date).dt.days

In [55]:
df

,applicants_applications_count,content_types,deal_id,main_image,min_social_media_followers,deal_tags,live_since,first_application_at,last_application_at,company_locations,...,tags,gender,featured_image,company_id,partner_id,diff_created_deleted,diff_updated_deleted,text_word_count,title_word_count,days_since_start
0,7,"[{'id': 38, 'name': 'Music', 'slug': 'Music-No...",019689e3-09ed-00c8-ada6-052b1041b584,uploads/deals/019689e3-0a14-ffff-1130-d620d8c2...,2500,None,2023-09-21 07:40:56.211091,2023-09-24 21:36:08.063257,2023-10-05 17:01:53.664059,"[{""id"": ""01KERHSC8805FW4D6D5GG90JMF"", ""name"": ...",...,None,None,None,019bb11c-b0d3-015e-baaf-b9fa5b347794,01992995-1769-0065-49aa-6a67b7f7af7d,NaT,843 days 23:58:36.025130,10,5,13
1,0,"[{'id': 23, 'name': 'Food', 'slug': 'Pizza'}]",019bbd3e-f64e-00c8-0e08-32e34caaaef2,uploads/deals/019bbd3e-75fd-ffff-9bca-d9682beb...,2500,None,2026-01-14 16:02:58.879237,NaT,NaT,"[{""id"": ""01KERHSGAG05FMYGV8JYE2PEAT"", ""name"": ...",...,None,unisex,uploads/deals/featured/019bbd3e-97e0-ffff-3597...,019bb11c-c121-015e-13c7-cd9c8b1c88e4,0199714c-8971-0065-43d5-fca6c121ee7b,NaT,0 days 00:00:00.114981,1,7,860
2,56,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip...",019a118c-fcd5-00c8-6a5c-2b771b7f18a0,uploads/deals/019a118c-a436-ffff-eaa7-e9db6105...,2500,None,2025-10-23 18:25:27.486337,2025-10-23 18:46:51.044149,2025-12-17 17:00:51.616012,None,...,None,None,None,None,01992996-8f9a-0065-30a1-aa9519cd8e27,NaT,0 days 03:34:43.689299,21,6,777
3,0,"[{'id': 21, 'name': 'Fashion', 'slug': 'Dress'}]",019689e5-f489-00c8-8c26-06b3ea0961a5,uploads/deals/019689e5-f54b-ffff-dbed-637e9d2a...,5000,None,2023-11-28 14:59:51.626780,NaT,NaT,None,...,None,None,None,None,01992995-2f32-0065-d8da-ac763ad1f309,NaT,519 days 12:31:19.973315,27,2,82
4,32,"[{'id': 23, 'name': 'Food', 'slug': 'Pizza'}, ...",019689e3-293c-00c8-c1df-735b603c5363,uploads/deals/019689e3-2aac-ffff-196b-7e667fbc...,10000,None,2024-05-22 12:00:33.784921,2024-05-22 12:08:26.158210,2024-06-28 09:56:33.026989,"[{""id"": ""01KERHVBR505FMVR4FSDTRR6AQ"", ""name"": ...",...,None,None,None,019bb11d-aee7-015e-7a46-e31c2aefb012,01992995-7e65-0065-edfb-434b238ddad4,NaT,599 days 19:38:59.828243,35,11,258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7272,23,"[{'id': 23, 'name': 'Food', 'slug': 'Pizza'}, ...",019a8e95-765c-00c8-3c2d-c82d439d6798,uploads/deals/019a8e8d-91f5-ffff-08c8-be10fa47...,1500,None,2025-11-16 21:32:31.363091,2025-11-16 22:52:14.069908,2026-01-21 06:00:18.685408,"[{""id"": ""01KERJ12X905FNHCM2X7CBZMR3"", ""name"": ...",...,None,None,None,019bb120-8b88-015e-c255-5acb9ca19b34,01992996-00e9-0065-81ca-dbae66b7d097,NaT,56 days 10:07:02.679972,63,4,801
7273,6,"[{'id': 35, 'name': 'Mom', 'slug': 'Baby'}]",019bc149-164d-00c8-3fd0-4ba01f6a2d70,uploads/deals/019bc148-d932-ffff-2c30-8f1668c6...,1500,None,2026-01-20 13:14:26.151491,2026-01-20 23:53:40.243580,2026-01-27 22:06:39.486637,None,...,None,unisex,uploads/deals/featured/019bc149-077a-ffff-5a8b...,019bb7c5-325c-015e-5c9a-3447bec38f70,019b2165-c3cb-0065-2ac7-05958d3ba823,NaT,5 days 02:22:19.044821,9,6,861
7274,41,"[{'id': 20, 'name': 'Beauty', 'slug': 'Heart'}...",019adef8-be38-00c8-79d3-7661c386cf03,uploads/deals/019adef8-a53d-ffff-95a6-fb7e39a9...,5000,None,2025-12-02 12:10:35.116208,2025-12-02 13:29:52.214366,2026-01-14 13:22:40.964880,None,...,None,None,None,None,019ad919-4b0a-0065-06f7-821b102edea9,NaT,0 days 00:03:03.731352,100,7,817
7275,7,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019bbcee-7663-00c8-dc81-a59920160cfb,uploads/deals/019bbd1a-f01c-ffff-7bb5-d438a57a...,2500,None,2026-01-14 15:28:22.254708,2026-01-15 09:49:58.259999,2026-01-22 10:51:52.051144,None,...,None,unisex,uploads/deals/featured/019bbd1a-bc64-ffff-45e0...,019bbc6a-85c8-015e-6a64-76b39f2df545,019bbc55-4368-0065-222c-b7a23f0b8eb2,NaT,17 days 08:28:55.137698,1,4,860


Remove 'test' deals (this is too aggressive, removes valid deals as well)

In [56]:
# print(f"Number of rows before cleaning: {len(df)}")
# mask_test = df['deal_title'].str.contains('test') | df['deal_text'].str.contains('test') | df['creators_requirement'].str.contains('test')
# df_containing_test = df[mask_test]
# df = df[~mask_test]

# print(f"Number of rows after cleaning: {len(df)}")

## Features

- Deal language

In [57]:
# from langdetect import detect, detect_langs

# deal_langs = []
# for i, deal in enumerate(df['deal_text']):
#     try:
#         lang = detect(deal)
#         deal_langs.append(lang)
#     except Exception as e:
#         print(e)
#         deal_langs.append(np.nan)

# df['deal_language'] = deal_langs

# # Afrikaans is actually Dutch
# df.loc[df.deal_language == 'af','deal_language'] = 'nl'

# # Filter languages that are not Dutch, English or German (they are gibberish)
# allowed_languages = ['nl', 'en', 'de']
# test = df[~df['deal_language'].isin(allowed_languages)]

# df = df[df['deal_language'].isin(allowed_languages)]
# len(df)

## Save

In [58]:
processed_data_path = paths.PROCESSED_DATA_DIR / 'BARTER_DEALS_CLEAN.parquet'
df.to_parquet(processed_data_path)

--------

# EDA

In [ ]:
df_zeroapps = df[df.applicants_applications_count == 0]
df_zeroapps


In [41]:
df.groupby('partner_id')['applicants_applications_count'].agg(['mean', 'count'])


,mean,count
partner_id,,
01992995-169b-0065-5f8a-63ee83969775,3.00,1
01992995-16cb-0065-1d18-645207443ac1,5.25,8
01992995-1702-0065-b840-710bae2b98cc,79.00,3
01992995-1731-0065-dd54-87d2a9b63463,152.50,4
01992995-1769-0065-49aa-6a67b7f7af7d,18.00,8
...,...,...
019bf568-9f9b-0065-5e2c-4c8ee15753a7,34.00,1
019bfab9-b4b6-0065-55ec-fd984cb40439,22.00,2
019c04df-a757-0065-6633-9b03e6247654,2.00,1


## Deal activity logs

In [22]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

import numpy as np
import pandas as pd

In [23]:
# Load environment variables from .env file
load_dotenv("../private_data/.env")

host = os.getenv("HOST")
db   = os.getenv("DB")
port   = os.getenv("PORT")
role = os.getenv("ROLE")
pw   = os.getenv("PASSWORD")
engine = create_engine(f"postgresql+psycopg2://{role}:{pw}@{host}:{port}/{db}")


- Deal logs: check difference between "before" and "after"

In [27]:
query = """
SELECT * FROM public.deal_activity_logs
WHERE entity_type = 'deal';
"""
deal_logs = pd.read_sql(query, engine)


In [31]:
deal_logs.entity_id.nunique()

7466

In [37]:
import pandas as pd
import json

def get_json_diff(before_json, after_json):
    # Handle NaNs or empty strings gracefully
    if pd.isna(before_json): before_json = "{}"
    if pd.isna(after_json): after_json = "{}"
    
    # Parse JSON strings into Python dictionaries
    # (If your columns are already dicts, you can skip the json.loads part)
    try:
        b_dict = json.loads(before_json) if isinstance(before_json, str) else before_json
        a_dict = json.loads(after_json) if isinstance(after_json, str) else after_json
    except json.JSONDecodeError:
        return {} # Return empty if parsing fails

    def _diff(b, a):
        changes = {}
        # Iterate through the 'after' state to find additions or updates
        for key, after_val in a.items():
            if key not in b:
                # 1. New key added
                changes[key] = after_val 
            elif isinstance(after_val, dict) and isinstance(b[key], dict):
                # 2. Both are nested dictionaries: Recurse!
                nested_diff = _diff(b[key], after_val)
                if nested_diff: # Only add if there were actually changes inside
                    changes[key] = nested_diff
            elif b[key] != after_val:
                # 3. Value changed (e.g., 'draft' -> 'live')
                changes[key] = after_val
                
        # Optional: You can also loop through `b` to find deleted keys
        # but for status changes, the above is usually sufficient.
        return changes

    return _diff(b_dict, a_dict)

In [ ]:
# Create the new column containing only the nested changes
deal_logs['changes'] = deal_logs.apply(
    lambda row: get_json_diff(row['before'], row['after']), 
    axis=1
)

- Deals with bad performance compared to expected deals (based on partner avg apps)

In [45]:
import pandas as pd

def flag_low_performing_deals(df, low_app_threshold=10, min_historical_avg=20, min_deals_run=3):
    """
    Flags deals that received < 10 apps, BUT ONLY IF the partner 
    historically performs well and has a proven track record.
    """
    # 1. Calculate partner historical stats
    partner_stats = df.groupby('partner_id')['applicants_applications_count'].agg(
        partner_avg_apps='mean',
        partner_deal_count='count'
    ).reset_index()
    
    # 2. Merge the historical stats back to the main dataframe
    df = df.merge(partner_stats, on='partner_id', how='left')
    
    # 3. Apply the Flagging Logic
    df['is_unexpectedly_low'] = (
        (df['applicants_applications_count'] < low_app_threshold) & # Has fewer than 10 apps
        (df['partner_avg_apps'] >= min_historical_avg) &            # Partner USUALLY gets 15+
        (df['partner_deal_count'] >= min_deals_run)                 # Partner has run at least 3 deals
    )
    
    return df

# Apply the function
df_flagged = flag_low_performing_deals(df)

# View only the anomalous deals
anomalies = df_flagged[df_flagged['is_unexpectedly_low'] == True]

In [46]:
anomalies

,applicants_applications_count,content_types,deal_id,main_image,min_social_media_followers,deal_tags,live_since,first_application_at,last_application_at,company_locations,...,featured_image,company_id,partner_id,diff_created_deleted,diff_updated_deleted,text_word_count,title_word_count,partner_avg_apps,partner_deal_count,is_unexpectedly_low
118,0,"[{'id': 21, 'name': 'Fashion', 'slug': 'Dress'}]",019689e6-5815-00c8-27fa-44d90aef8aec,uploads/deals/019689e6-5951-ffff-d7b9-c3d58138...,5000,None,2025-02-14 05:55:26.632305,NaT,NaT,None,...,None,None,01992995-b483-0065-5172-4fcb85632a19,NaT,75 days 21:36:10.886400,43,5,20.333333,3,True
120,0,"[{'id': 23, 'name': 'Food', 'slug': 'Pizza'}]",019689e4-ce6b-00c8-ee49-d2b323a4806c,uploads/deals/019689e4-cf63-ffff-767c-fe0a6850...,2500,None,2025-04-29 07:59:40.130194,NaT,NaT,"[{""id"": ""01KERJ26D105FM72RARYSMA2AW"", ""name"": ...",...,None,019bb121-1941-015e-7c19-e040d64cbd2a,01992996-0c9f-0065-72a4-45ad4bbc1231,NaT,257 days 23:39:38.822039,47,6,20.312500,16,True
143,3,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019689e1-e3b3-00c8-60a3-6ff40abe4535,uploads/deals/019689e1-e491-ffff-67c5-3bbfee3e...,5000,None,2025-02-21 12:14:54.016826,2025-02-21 13:06:56.734867,2025-03-03 16:50:00.721277,"[{""id"": ""01KERJ1Q0A05FXRKWE6K9CX0YM"", ""name"": ...",...,None,019bb120-dbc8-015e-51fc-db0cd4579c16,01992996-0869-0065-344e-587d089c5563,NaT,324 days 19:24:22.941058,110,3,20.076923,13,True
166,8,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019b30d8-db23-00c8-63f9-cc28b76ebe81,uploads/deals/019b4b04-f022-ffff-73df-827a6194...,2500,None,2025-12-18 09:44:37.066760,2025-12-18 10:07:23.079156,2025-12-28 19:21:23.085027,"[{""id"": ""01KERHTV7Y05FK215SFDKWW43A"", ""name"": ...",...,None,019bb11d-6ae7-015e-dd34-32ef9d3f651d,01998128-6973-0065-0e0b-b8f0e298afa5,NaT,26 days 05:36:02.786190,116,7,23.833333,6,True
204,6,"[{'id': 20, 'name': 'Beauty', 'slug': 'Heart'}]",019689e5-0d39-00c8-8b71-72dbb55a1d34,uploads/deals/019689e5-0dfa-ffff-2809-d2b1b3f6...,1500,None,NaT,2025-02-07 14:07:44.896181,2025-02-07 16:36:16.235022,None,...,None,None,01992995-ee85-0065-1893-3afa5a9dd97e,85 days 04:01:58.852775,82 days 14:58:46.795875,92,6,42.653846,52,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6334,7,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",01981c4e-0f86-00c8-e451-cde0c5503496,uploads/deals/01981c4c-6ed2-ffff-41ad-f7c7c478...,1500,None,2025-07-21 12:10:14.138950,2025-07-22 07:47:53.402937,2025-12-23 13:21:20.717142,None,...,None,None,01992996-bffe-0065-19db-89673244a5cf,NaT,201 days 21:07:56.762971,52,2,26.083333,12,True
6342,5,"[{'id': 22, 'name': 'Travel', 'slug': 'Airplan...",019c08ec-6ea1-00c8-51b8-e48c90a511ae,uploads/deals/019c08ea-504d-ffff-cc27-ffbbf7ce...,5000,None,2026-01-29 08:43:58.593701,2026-01-29 10:00:20.390261,2026-01-29 18:11:48.960160,None,...,uploads/deals/featured/019c08ea-85d2-ffff-5790...,019bb11f-3f75-015e-1178-35d11b58e53a,01992996-3732-0065-2ae8-b0e12bbc1a6a,NaT,0 days 00:01:34.992762,1,9,60.500000,8,True
6396,8,"[{'id': 37, 'name': 'Luxury', 'slug': 'Sketch-...",019b4665-e9b2-00c8-7599-8b1581c8ab61,uploads/deals/019b465f-4d49-ffff-61a1-54d9baa2...,2500,None,2025-12-22 14:10:42.878785,2025-12-22 20:46:45.906529,2026-01-10 11:56:09.344348,None,...,None,None,01992996-7665-0065-d9c7-27e8606f3c88,NaT,0 days 00:00:00.157675,169,8,61.250000,4,True
6408,4,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019bb9c6-30cc-00c8-eaa5-37d1b9372497,uploads/deals/featured/019bb9c6-31b8-ffff-c92b...,1500,None,2026-01-13 23:52:30.813243,2026-01-18 13:20:19.714824,2026-01-22 14:40:32.544884,None,...,uploads/deals/019bb9c6-33f8-ffff-14b8-7610fd12...,019bb120-3341-015e-265e-59d6c527da88,01992995-57cf-0065-a3a6-10cdb1dc8c1b,NaT,0 days 00:00:31.363432,1,10,22.666667,30,True
